# PRIX: re-run from a published snapshot (Python, offline)

This notebook needs only the Python standard library. It:
1. loads `data/snapshots/<date>.json` and checks its `sha256` (canonical JSON, sorted keys, as in `lib/snapshot/hash.ts`);
2. recomputes three verdict inputs independently of the TypeScript engine: the H2 custody/TVL ratio, H3's count of tokens with $100k impact of 1% or less, and H6's recovery ratios.

The authoritative engine is `lib/thesis/engine.ts` (`pnpm verdicts`). This notebook is a cross-check.
Thresholds: `thresholds/v1.yaml`, **DRAFT, not locked**.

In [ ]:
import json, hashlib, glob, statistics
from decimal import Decimal

SNAP = sorted(glob.glob("../data/snapshots/????-??-??.json"))[-1]
s = json.load(open(SNAP))
print(SNAP, s["as_of"])

In [ ]:
def jsnum(x):
    # Number formatting identical to JavaScript's JSON.stringify
    if isinstance(x, bool): return "true" if x else "false"
    if isinstance(x, int): return str(x)
    if x == int(x) and abs(x) < 1e21: return str(int(x))
    r = repr(x)
    if 1e-7 <= abs(x) < 1e21:
        return format(Decimal(r), "f")
    m, e = r.split("e"); e = int(e)
    return f"{m}e{'+' if e > 0 else '-'}{abs(e)}"

def canonical(v):
    if v is None: return "null"
    if isinstance(v, bool): return "true" if v else "false"
    if isinstance(v, (int, float)): return jsnum(v)
    if isinstance(v, str): return json.dumps(v, ensure_ascii=False)
    if isinstance(v, list): return "[" + ",".join(canonical(x) for x in v) + "]"
    return "{" + ",".join(f"{json.dumps(k, ensure_ascii=False)}:{canonical(v[k])}" for k in sorted(v)) + "}"

h = hashlib.sha256(canonical({k: v for k, v in s.items() if k != "sha256"}).encode()).hexdigest()
print("sha256 OK" if h == s["sha256"] else f"MISMATCH {h} != {s['sha256']}")

In [ ]:
# H2: custody / clean RHEA TVL, per daily snapshot (needs 30 days for a verdict)
for d in s["series"]["h2_daily"]:
    print(d["date"], f"{d['ratio']:.2f}x", "(>= 2x)" if d["ratio"] >= 2 else "")
print(f"{len(s['series']['h2_daily'])} of 30 days in the window")

In [ ]:
# H3: non-NEAR tokens whose best-route $100k relative slippage is <= 1%
best = {}
for r in s["depth"]:
    if r["size_usd"] == 100000 and r["status"] == "ok" and r["relative_slippage"] is not None and r["token_id"] not in ("wrap.near", "near"):
        best[r["token_id"]] = min(best.get(r["token_id"], 1e9), r["relative_slippage"])
passing = sorted(t for t, v in best.items() if v <= 0.01)
sym = {x["token_id"]: x["symbol"] for x in s["index"]}
print(len(passing), [sym.get(t, t) for t in passing], "-> Supported if >= 5, Not supported if <= 1")

In [ ]:
# H6: day-30 values vs the 7-day pre-incident median
D = s["incident"]["date"]
def add(d, n):
    import datetime as dt
    return (dt.date.fromisoformat(d) + dt.timedelta(days=n)).isoformat()
cust = {d["date"]: d["usd"] for d in s["series"]["custody_constant"]}
txns = {d["date"]: d["txns"] for d in s["series"]["intents_txns"]}
base = [add(D, -i) for i in range(7, 0, -1)]
at = add(D, 30)
for name, m in (("custody (constant prices)", cust), ("intents.near txns", txns)):
    b = statistics.median([m[d] for d in base if d in m])
    print(f"{name}: baseline {b:,.0f}, day 30 ({at}) {m.get(at, float('nan')):,.0f} -> {m.get(at, float('nan')) / b:.0%}")
print("Sources:", *s["incident"]["sources"], sep="\n  ")